# Ноутбук 2 — Извлечение признаков

**Назначение.** Построение признаковой матрицы из сырых Frame-данных:
2-секундные окна с 50%-перекрытием, окуломоторные и физиологические признаки,
9 типов статистик каждого канала.

**Входы.** Frame-данные датасета (после ноутбука 1).

**Выходы.** `source/results/feature_table.csv` (15 104 окна × 145 признаков).


In [1]:
import sys, time
from pathlib import Path

NB_ROOT = Path.cwd()
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modules import config, features

print(f"Notebooks root: {NB_ROOT}")
print(f"Датасет: {config.DATASET_ROOT}")
print(f"Результаты: {config.RESULTS_DIR}")

Notebooks root: D:\Programming\Python\Diplom\notebooks
Датасет: D:\Programming\Python\Diplom\CEAP-360VR-Dataset-master\CEAP-360VR
Результаты: D:\Programming\Python\Diplom\source\results


## 2.1 Параметры извлечения

Окно 2 секунды с 50%-перекрытием.
При длительности видео 60 с это даёт 59 окон на видео; при 8 видео и 32 участниках —
15 104 окна суммарно.

In [2]:
print(f"Размер окна: {config.WINDOW_SIZE_SEC} с")
print(f"Перекрытие:  {config.WINDOW_OVERLAP * 100:.0f}%")
print(f"Частота:     {config.FRAME_RATE_HZ} Гц")
print(f"Порог отбраковки окна по невалидности айтрекинга: "
      f"{config.MAX_INVALID_EYE_RATIO * 100:.0f}%")

Размер окна: 2 с
Перекрытие:  50%
Частота:     25 Гц
Порог отбраковки окна по невалидности айтрекинга: 30%


## 2.2 Запуск пайплайна построения матрицы признаков

`modules.features` инкапсулирует полный цикл: загрузка по каждому участнику,
окно за окном, извлечение признаков по двум модальностям, агрегация,
формирование таргетов (`arousal_class_bin`, `ssq_high` и др.).

In [3]:
feature_table_path = config.RESULTS_DIR / "feature_table.csv"

if feature_table_path.exists():
    print(f"Загружаю готовую таблицу: {feature_table_path}")
    df = pd.read_csv(feature_table_path)
else:
    print("Запускаю построение матрицы признаков (~2 мин)...")
    all_rows = []
    t0 = time.time()
    for pid in config.PARTICIPANT_IDS:
        ts = time.time()
        try:
            rows = features.extract_all_features_for_participant(pid)
            all_rows.extend(rows)
            print(f"  {pid}: {len(rows)} окон ({time.time()-ts:.1f} с)")
        except Exception as e:
            print(f"  {pid}: ОШИБКА — {e}")
    df = pd.DataFrame(all_rows)
    df = features.add_global_targets(df)
    df.to_csv(feature_table_path, index=False)
    print(f"\nГотово за {time.time()-t0:.1f} с. Сохранено: {feature_table_path}")

print(f"Размер: {df.shape}")
df.head(3)

Загружаю готовую таблицу: D:\Programming\Python\Diplom\source\results\feature_table.csv
Размер: (15104, 156)


,participant,video,window_idx,t_start,t_end,left_pupil_mean,left_pupil_std,left_pupil_min,left_pupil_max,left_pupil_median,...,hrv_rmssd,hrv_pnn50,arousal_mean,valence_mean,ssq_post_total,ssq_pre_total,ssq_delta,arousal_class_bin,arousal_class_tri,ssq_high
0,P1,V1,0,0.0,2.0,4.97994,0.887570,3.377,5.706,5.567,...,67.435487,0.500000,5.0,5.0,7.48,26.18,-18.7,0,1,0
1,P1,V1,1,1.0,3.0,3.88748,0.795956,3.263,5.706,3.489,...,42.314669,0.333333,5.0,5.0,7.48,26.18,-18.7,0,1,0
2,P1,V1,2,2.0,4.0,3.24250,0.193372,2.871,3.613,3.248,...,40.190173,0.153846,5.0,5.0,7.48,26.18,-18.7,0,1,0


## 2.3 Состав признаков

In [4]:
META = {
    "participant", "video", "window_idx", "t_start", "t_end",
    "arousal_mean", "valence_mean",
    "ssq_post_total", "ssq_pre_total", "ssq_delta",
    "arousal_class_bin", "arousal_class_tri", "ssq_high",
}
feature_cols = [c for c in df.columns if c not in META]
print(f"Признаков: {len(feature_cols)}")

def channel_of(col: str) -> str:
    if col.startswith(("left_pupil", "right_pupil")):
        return "pupil"
    if col.startswith("gaze"):
        return "gaze"
    if col.startswith("head"):
        return "head"
    if col.startswith("eda"):
        return "eda"
    if col.startswith("hrv"):
        return "hrv"
    if col.startswith(("hr_", "hr_mean", "hr_std")):
        return "hr"
    if col.startswith("skt"):
        return "skt"
    if col.startswith("bvp"):
        return "bvp"
    if col.startswith("acc"):
        return "acc"
    if col.startswith("ibi"):
        return "ibi"
    return "other"

channels = pd.Series([channel_of(c) for c in feature_cols]).value_counts()
print("\nПризнаков по каналам:")
print(channels.to_string())

Признаков: 143

Признаков по каналам:
pupil    27
gaze     27
head     27
eda      11
skt      10
other     9
hr        9
bvp       9
acc       9
hrv       5


## 2.4 Распределение таргетов

In [5]:
print("Бинарный arousal (per-window, по глобальной медиане):")
print(df["arousal_class_bin"].value_counts().to_string())

print("\nSSQ > 15 (block-level, по участнику):")
ssq_per_pid = df.groupby("participant")["ssq_high"].first()
print(ssq_per_pid.value_counts().to_string())

Бинарный arousal (per-window, по глобальной медиане):
arousal_class_bin
0    8762
1    6342

SSQ > 15 (block-level, по участнику):
ssq_high
0    18
1    14


## 2.5 Выводы

* Получена матрица 15 104 × 145, готовая к использованию во всех экспериментах.
* Приблизительный баланс таргетов: `arousal_class_bin` распределён 57/43
  (низ/высок по глобальной медиане); `SSQ > 15` на уровне участника — 14/18 (44%/56%).
* Признаки распределены примерно как 90 окуломоторных и 53 физиологических.

Дальнейший шаг — разведочный анализ распределений и корреляций (ноутбук 3).